# Comprehensive Scaling Analysis - Report Figures Generator

This notebook provides an in-depth analysis of the parallel performance of a 5-point stencil computation (Heat Equation solver) using MPI and OpenMP parallelization. It generates all figures used in the REPORT.md document and provides detailed statistical analysis.

## Analysis Scope

This comprehensive study examines:

1. **OpenMP Scaling** - Shared memory parallelization efficiency across different thread counts
2. **Strong Scaling** - Fixed problem size with varying computational resources  
3. **Weak Scaling** - Proportional problem size scaling with resources
4. **Communication Overhead** - Analysis of parallel communication costs
5. **Configuration Comparison** - Impact of different task/thread distributions
6. **Measurement Quality** - Statistical analysis of measurement reliability

## Methodology

Performance is evaluated using standard metrics:
- **Speedup**: $S_p = \frac{T_1}{T_p}$ where $T_1$ is baseline time and $T_p$ is time with p resources
- **Efficiency**: $E_p = \frac{S_p}{p} \times 100\%$ (ideally ≈ 100% for perfect scaling)
- **Parallel Overhead**: Time spent in communication and synchronization
- **Scalability**: Ability to maintain efficiency as resources increase

## Figures Generated:

1. **OpenMP Scaling**: `omp_speedup.png`, `omp_efficiency.png`
2. **Strong Scaling**: `strong_speedup_efficiency.png`, `strong_execution_time.png`
3. **Weak Scaling**: `weak_efficiency.png`, `weak_execution_time.png`

All figures are saved to the `figures/` directory.

## 1. Setup and Imports

## 2. OpenMP Scaling Analysis

OpenMP scaling measures the performance improvement when increasing the number of threads within a single MPI process on a single node. This analysis evaluates shared-memory parallelization efficiency across different thread counts (1-112 threads).

**Key Metrics:**
- **Speedup**: $S_p = \frac{T_1}{T_p}$ where $T_1$ is the baseline time with 1 thread and $T_p$ is time with $p$ threads
- **Efficiency**: $E_p = \frac{S_p}{p}$ (ideally ≈ 1.0 for perfect scaling)

The analysis helps identify the optimal number of threads per MPI task for hybrid parallelization.

In [ ]:
import sys
import os
from pathlib import Path

# Add scripts directory to path to import plotting functions
script_dir = Path.cwd() / 'scripts' / 'plots_used'
project_root = script_dir.parent.parent
sys.path.insert(0, str(script_dir))

# Import plotting functions from scripts
from plot_omp_speedup import load_and_process_data as load_omp_data, plot_omp_speedup
from plot_omp_efficiency import load_and_process_data as load_omp_eff_data, plot_omp_efficiency
from plot_strong_speedup_efficiency import load_and_process_data as load_strong_data, plot_strong_speedup_efficiency
from plot_strong_execution_time import load_and_process_data as load_strong_exec_data, plot_strong_execution_time
from plot_weak_scaling import load_and_process_data as load_weak_data, plot_weak_scaling
from plot_weak_execution_time import load_and_process_data as load_weak_exec_data, plot_weak_execution_time

# Configure paths
figures_dir = project_root / 'figures'
figures_dir.mkdir(exist_ok=True)

print(f"✓ Project root: {project_root}")
print(f"✓ Scripts directory: {script_dir}")
print(f"✓ Figures directory: {figures_dir}")

## 2. OpenMP Scaling Analysis

### 2.1 OpenMP Speedup

In [ ]:
# Load OpenMP data
omp_csv = project_root / 'data' / 'SM3800083_omp_serial_results.csv'
print(f"Loading OpenMP data from: {omp_csv}")
omp_results = load_omp_data(str(omp_csv))

# Generate speedup plot
output_path = figures_dir / 'omp_speedup.png'
print(f"\nGenerating OpenMP speedup plot...")
plot_omp_speedup(omp_results, str(output_path))
print(f"✓ Saved to: {output_path}")

## 3. Strong Scaling Analysis

Strong scaling measures performance when the problem size remains fixed (16384×16384 grid) while increasing the number of computational nodes. Ideal strong scaling shows linear speedup, but is limited by Amdahl's Law and communication overhead.

**Key Metrics:**
- **Speedup**: $S_p = \frac{T_1}{T_p}$ where $T_1$ is baseline time on 1 node and $T_p$ is time with $p$ nodes
- **Efficiency**: $E_p = \frac{S_p}{p} \times 100\%$ (ideally ≈ 100% for perfect scaling)
- **Communication Overhead**: Percentage of total time spent in MPI communication

This analysis compares three hybrid configurations (8×14, 2×56, 16×7) to identify the optimal balance between MPI tasks and OpenMP threads.

### 2.2 OpenMP Efficiency

In [ ]:
# Generate efficiency plot
omp_eff_results = load_omp_eff_data(str(omp_csv))
output_path = figures_dir / 'omp_efficiency.png'
print(f"\nGenerating OpenMP efficiency plot...")
plot_omp_efficiency(omp_eff_results, str(output_path))
print(f"✓ Saved to: {output_path}")

## 3. Strong Scaling Analysis

### 3.1 Strong Scaling Speedup and Efficiency

In [ ]:
# Load strong scaling data
strong_csv = project_root / 'data' / 'SM3800083_strong_parallel_results.csv'
print(f"Loading strong scaling data from: {strong_csv}")
strong_results = load_strong_data(str(strong_csv))

# Generate combined speedup and efficiency plots
output_speedup = figures_dir / 'strong_speedup.png'
output_efficiency = figures_dir / 'strong_efficiency.png'
output_combined = figures_dir / 'strong_speedup_efficiency.png'

print(f"\nGenerating strong scaling plots...")
plot_strong_speedup_efficiency(
    strong_results,
    str(output_speedup),
    str(output_efficiency),
    str(output_combined)
)
print(f"✓ Saved to: {output_combined}")

## 4. Weak Scaling Analysis

Weak scaling measures performance when both the problem size and computational resources increase proportionally, maintaining constant work per core. Ideal weak scaling shows constant execution time (100% efficiency) as resources scale.

**Key Metrics:**
- **Runtime**: Should remain constant across node counts for ideal weak scaling
- **Efficiency**: $E_p = \frac{T_1}{T_p} \times 100\%$ where $T_1$ and $T_p$ are runtimes at baseline and $p$ nodes
- **Grid Scaling**: Problem size scales proportionally (16384×16384 → 65536×65536)

This analysis evaluates both **non-periodic** (fixed) and **periodic** (wrap-around) boundary conditions to assess their impact on scalability and communication overhead.

### 3.2 Strong Scaling Execution Time Breakdown

In [ ]:
# Load execution time data
strong_exec_results = load_strong_exec_data(str(strong_csv), build_variant='ofast')

# Generate execution time plot
output_path = figures_dir / 'strong_execution_time.png'
print(f"\nGenerating strong scaling execution time plot...")
plot_strong_execution_time(strong_exec_results, str(output_path))
print(f"✓ Saved to: {output_path}")

## 4. Weak Scaling Analysis

### 4.1 Weak Scaling Efficiency

In [ ]:
# Load weak scaling data (corrected CSV)
weak_csv = project_root / 'data' / 'SM3800083_weak_parallel_results_corrected.csv'
print(f"Loading weak scaling data from: {weak_csv}")
weak_results = load_weak_data(str(weak_csv), build_variant='ofast_omp_improved')

# Generate efficiency plot
output_path = figures_dir / 'weak_efficiency.png'
print(f"\nGenerating weak scaling efficiency plot...")
plot_weak_scaling(weak_results, str(output_path))
print(f"✓ Saved to: {output_path}")

### 4.2 Weak Scaling Execution Time Breakdown

## 6. Data Analysis and Insights

This section provides detailed statistical analysis and insights from the performance data, including measurement quality assessment, performance metrics summary, and communication overhead analysis.

In [ ]:
# Load weak scaling execution time data
weak_exec_results = load_weak_exec_data(str(weak_csv), build_variant='ofast_omp_improved')

# Generate execution time plot
output_path = figures_dir / 'weak_execution_time.png'
print(f"\nGenerating weak scaling execution time plot...")
plot_weak_execution_time(weak_exec_results, str(output_path))
print(f"✓ Saved to: {output_path}")

## 5. Summary

In [ ]:
print("\n" + "="*80)
print("GENERATED FIGURES SUMMARY")
print("="*80)

expected_figures = [
    'omp_speedup.png',
    'omp_efficiency.png',
    'strong_speedup_efficiency.png',
    'strong_execution_time.png',
    'weak_efficiency.png',
    'weak_execution_time.png'
]

print(f"\nExpected figures in {figures_dir}:")
for fig_name in expected_figures:
    fig_path = figures_dir / fig_name
    status = "✓" if fig_path.exists() else "✗ MISSING"
    size = fig_path.stat().st_size if fig_path.exists() else 0
    size_kb = size / 1024
    print(f"  {status} {fig_name} ({size_kb:.1f} KB)")

print("\n" + "="*80)
print("All report figures have been generated successfully!")
print("="*80)

## 6. Data Analysis and Insights

This section provides detailed statistical analysis and insights from the performance data.

In [ ]:
import pandas as pd
import numpy as np

# Load all datasets for analysis
omp_df = pd.read_csv(project_root / 'data' / 'SM3800083_omp_serial_results.csv')
strong_df = pd.read_csv(project_root / 'data' / 'SM3800083_strong_parallel_results.csv')
weak_df = pd.read_csv(project_root / 'data' / 'SM3800083_weak_parallel_results_corrected.csv')

print("=" * 80)
print("DATASET SUMMARY")
print("=" * 80)
print(f"\n📊 OpenMP Data: {len(omp_df)} records")
print(f"   Build variants: {sorted(omp_df['BuildVariant'].unique())}")
print(f"   Thread counts: {sorted(omp_df['Threads'].unique())}")
print(f"   Energy sources: {sorted(omp_df['EnergySources'].unique())}")

print(f"\n📊 Strong Scaling Data: {len(strong_df)} records")
print(f"   Build variants: {sorted(strong_df['BuildVariant'].unique())}")
print(f"   Node counts: {sorted(strong_df['Nodes'].unique())}")
print(f"   Configurations: {len(strong_df.groupby(['TasksPerNode', 'ThreadsPerTask']))} unique")
print(f"   Energy sources: {sorted(strong_df['EnergySources'].unique())}")

print(f"\n📊 Weak Scaling Data: {len(weak_df)} records")
print(f"   Build variants: {sorted(weak_df['BuildVariant'].unique())}")
print(f"   Node counts: {sorted(weak_df['Nodes'].unique())}")
print(f"   Periodic boundaries: {sorted(weak_df['Periodic'].unique())}")
print(f"   Grid scaling: {weak_df.groupby('Nodes')[['XDim', 'YDim']].first().to_dict('index')}")

### 6.1 Measurement Quality Analysis

In [ ]:
def analyze_measurement_quality(data, key_cols, dataset_name):
    """Analyze measurement quality for duplicate measurements."""
    print("=" * 80)
    print(f"MEASUREMENT QUALITY: {dataset_name}")
    print("=" * 80)
    
    # Find duplicates
    grouped = data.groupby(key_cols)
    duplicates = data.groupby(key_cols).filter(lambda x: len(x) > 1)
    
    if len(duplicates) == 0:
        print(f"\n✓ No duplicate measurements found")
        return None
    
    stats_list = []
    for name, group in grouped:
        if len(group) > 1:
            values = group['TotalTime'].values
            mean_val = np.mean(values)
            std_val = np.std(values, ddof=1) if len(values) > 1 else 0
            cv = (std_val / mean_val * 100) if mean_val > 0 else 0
            percent_diff = (np.max(values) - np.min(values)) / mean_val * 100
            
            stats_list.append({
                'N_measurements': len(group),
                'Mean_TotalTime': mean_val,
                'CV_percent': cv,
                'Percent_Diff': percent_diff
            })
    
    if not stats_list:
        print(f"\n✓ No duplicate measurements found")
        return None
    
    stats_df = pd.DataFrame(stats_list)
    print(f"\n📊 Found {len(stats_df)} groups with duplicate measurements")
    print(f"   Mean CV: {stats_df['CV_percent'].mean():.2f}%")
    print(f"   Max CV: {stats_df['CV_percent'].max():.2f}%")
    print(f"   Mean percent difference: {stats_df['Percent_Diff'].mean():.2f}%")
    
    # Quality assessment
    excellent = (stats_df['CV_percent'] < 1.0).sum()
    good = ((stats_df['CV_percent'] >= 1.0) & (stats_df['CV_percent'] < 3.0)).sum()
    acceptable = ((stats_df['CV_percent'] >= 3.0) & (stats_df['CV_percent'] < 5.0)).sum()
    poor = (stats_df['CV_percent'] >= 5.0).sum()
    
    print(f"\n🔍 QUALITY ASSESSMENT:")
    print(f"   Excellent (CV < 1%):   {excellent}/{len(stats_df)} ({excellent/len(stats_df)*100:.1f}%)")
    print(f"   Good (1% ≤ CV < 3%):   {good}/{len(stats_df)} ({good/len(stats_df)*100:.1f}%)")
    print(f"   Acceptable (3% ≤ CV < 5%): {acceptable}/{len(stats_df)} ({acceptable/len(stats_df)*100:.1f}%)")
    print(f"   Poor (CV ≥ 5%):        {poor}/{len(stats_df)} ({poor/len(stats_df)*100:.1f}%)")
    
    return stats_df

# Analyze each dataset
print("\n")
omp_quality = analyze_measurement_quality(omp_df, ['Threads', 'EnergySources', 'BuildVariant'], 'OpenMP')
print("\n")
strong_quality = analyze_measurement_quality(strong_df, ['Nodes', 'TasksPerNode', 'ThreadsPerTask', 'EnergySources', 'BuildVariant'], 'Strong Scaling')
print("\n")
weak_quality = analyze_measurement_quality(weak_df, ['Nodes', 'TasksPerNode', 'ThreadsPerTask', 'EnergySources', 'BuildVariant', 'Periodic'], 'Weak Scaling')

### 6.2 Performance Metrics Summary

In [ ]:
# Filter for primary build variant
PRIMARY_BUILD = 'ofast_omp_improved'
omp_primary = omp_df[omp_df['BuildVariant'] == PRIMARY_BUILD].copy()
strong_primary = strong_df[strong_df['BuildVariant'] == 'ofast'].copy()  # Strong uses 'ofast'
weak_primary = weak_df[weak_df['BuildVariant'] == PRIMARY_BUILD].copy()

print("=" * 80)
print("PERFORMANCE METRICS SUMMARY")
print("=" * 80)

# OpenMP Analysis
print("\n📊 OpenMP Scaling (BuildVariant='ofast_omp_improved'):")
omp_1thread = omp_primary[omp_primary['Threads'] == 1]['TotalTime'].mean()
omp_112threads = omp_primary[omp_primary['Threads'] == 112]['TotalTime'].mean()
omp_speedup = omp_1thread / omp_112threads
omp_efficiency = omp_speedup / 112 * 100

print(f"   1 thread baseline:   {omp_1thread:.2f}s")
print(f"   112 threads:         {omp_112threads:.2f}s")
print(f"   Speedup (112 threads): {omp_speedup:.2f}×")
print(f"   Efficiency (112 threads): {omp_efficiency:.1f}%")

# Strong Scaling Analysis
print("\n📊 Strong Scaling (BuildVariant='ofast'):")
strong_configs = {
    '16×7': {'TasksPerNode': 16, 'ThreadsPerTask': 7},
    '8×14': {'TasksPerNode': 8, 'ThreadsPerTask': 14},
    '2×56': {'TasksPerNode': 2, 'ThreadsPerTask': 56}
}

for config_name, params in strong_configs.items():
    config_data = strong_primary[
        (strong_primary['TasksPerNode'] == params['TasksPerNode']) &
        (strong_primary['ThreadsPerTask'] == params['ThreadsPerTask']) &
        (strong_primary['EnergySources'] == 1)
    ]
    
    if len(config_data) > 0:
        baseline_1node = config_data[config_data['Nodes'] == 1]['TotalTime'].mean()
        final_16nodes = config_data[config_data['Nodes'] == 16]['TotalTime'].mean()
        if pd.notna(baseline_1node) and pd.notna(final_16nodes) and final_16nodes > 0:
            speedup = baseline_1node / final_16nodes
            efficiency = speedup / 16 * 100
            print(f"\n   {config_name} Configuration:")
            print(f"      1 node:   {baseline_1node:.2f}s")
            print(f"      16 nodes: {final_16nodes:.2f}s")
            print(f"      Speedup:  {speedup:.2f}×")
            print(f"      Efficiency: {efficiency:.1f}%")

# Weak Scaling Analysis
print("\n📊 Weak Scaling (BuildVariant='ofast_omp_improved', 16×7 config):")
weak_16x7 = weak_primary[
    (weak_primary['TasksPerNode'] == 16) &
    (weak_primary['ThreadsPerTask'] == 7) &
    (weak_primary['EnergySources'] == 1)
]

for periodic_val in [0, 1]:
    periodic_name = 'Non-Periodic' if periodic_val == 0 else 'Periodic'
    periodic_data = weak_16x7[weak_16x7['Periodic'] == periodic_val]
    
    if len(periodic_data) > 0:
        baseline_1node = periodic_data[periodic_data['Nodes'] == 1]['TotalTime'].mean()
        final_16nodes = periodic_data[periodic_data['Nodes'] == 16]['TotalTime'].mean()
        if pd.notna(baseline_1node) and pd.notna(final_16nodes):
            efficiency = (baseline_1node / final_16nodes) * 100
            print(f"\n   {periodic_name}:")
            print(f"      1 node:   {baseline_1node:.2f}s")
            print(f"      16 nodes: {final_16nodes:.2f}s")
            print(f"      Efficiency: {efficiency:.1f}%")

print("\n" + "=" * 80)

In [ ]:
print("=" * 80)
print("COMMUNICATION OVERHEAD ANALYSIS")
print("=" * 80)

# Strong Scaling Communication Overhead
print("\n📊 Strong Scaling Communication Overhead:")
strong_with_comm = strong_primary[
    (strong_primary['EnergySources'] == 1) &
    (strong_primary['BuildVariant'] == 'ofast')
].copy()

if 'CommunicationTime' in strong_with_comm.columns and 'TotalTime' in strong_with_comm.columns:
    strong_with_comm['CommOverhead'] = (strong_with_comm['CommunicationTime'] / strong_with_comm['TotalTime']) * 100
    
    for config_name, params in strong_configs.items():
        config_data = strong_with_comm[
            (strong_with_comm['TasksPerNode'] == params['TasksPerNode']) &
            (strong_with_comm['ThreadsPerTask'] == params['ThreadsPerTask'])
        ]
        
        if len(config_data) > 0:
            avg_overhead = config_data['CommOverhead'].mean()
            max_overhead = config_data['CommOverhead'].max()
            print(f"\n   {config_name} Configuration:")
            print(f"      Average overhead: {avg_overhead:.2f}%")
            print(f"      Maximum overhead: {max_overhead:.2f}%")

# Weak Scaling Communication Overhead
print("\n📊 Weak Scaling Communication Overhead (16×7 config):")
weak_with_comm = weak_primary[
    (weak_primary['TasksPerNode'] == 16) &
    (weak_primary['ThreadsPerTask'] == 7) &
    (weak_primary['EnergySources'] == 1)
].copy()

if 'CommunicationTime' in weak_with_comm.columns and 'TotalTime' in weak_with_comm.columns:
    weak_with_comm['CommOverhead'] = (weak_with_comm['CommunicationTime'] / weak_with_comm['TotalTime']) * 100
    
    for periodic_val in [0, 1]:
        periodic_name = 'Non-Periodic' if periodic_val == 0 else 'Periodic'
        periodic_data = weak_with_comm[weak_with_comm['Periodic'] == periodic_val]
        
        if len(periodic_data) > 0:
            avg_overhead = periodic_data['CommOverhead'].mean()
            max_overhead = periodic_data['CommOverhead'].max()
            print(f"\n   {periodic_name}:")
            print(f"      Average overhead: {avg_overhead:.2f}%")
            print(f"      Maximum overhead: {max_overhead:.2f}%")

print("\n" + "=" * 80)